# RAG Strategy Evaluation — No RAGAS, Pure NIM

## Why No RAGAS?
RAGAS has three problems with your setup:
1. **Version chaos** — `0.1.x` vs `0.2.x` have completely different APIs and import paths
2. **Parallel calls** — RAGAS fires async LLM calls in bursts, which kills NIM free tier with 429s and hangs silently
3. **LangChain wrapper fragility** — NIM + LangChain + RAGAS is three layers of abstraction, each with its own breaking changes

## What We Do Instead
RAGAS is just cleverly written LLM prompts under the hood. We write those same prompts ourselves, call NIM directly with `time.sleep()` between calls, and get scores + reasoning we can actually read.

## Notebook Structure
```
Part A  →  Build test set (run ONCE, cached to JSON)
Part B  →  Run your pipelines on the test set (one per strategy)
Part C  →  NIM-as-Judge evaluation (4 metrics, sequential calls)
Part D  →  Results table for your capstone report
```

## The 4 Metrics
| Metric | Question it answers | Needs ground truth? |
|---|---|---|
| **Faithfulness** | Does the answer stick to what the docs say? | No |
| **Answer Relevancy** | Does the answer address the question? | No |
| **Context Precision** | Were all retrieved chunks actually useful? | Yes |
| **Context Recall** | Did we retrieve everything needed? | Yes |

---
## Setup — Imports & NIM Client

In [ ]:
import json
import os
import pickle
import random
import time
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(Path.home() / "Projects" / "RAG_v1" / ".env")
NVIDIA_API_KEY = os.environ["NVIDIA_API_KEY"]

# ── Single NIM client — used for EVERYTHING in this notebook ─────────────────
nim = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY,
)

GENERATOR_MODEL = "meta/llama-3.1-70b-instruct"   # for test-set generation
JUDGE_MODEL     = "meta/llama-3.1-70b-instruct"   # for evaluation scoring
# ─────────────────────────────────────────────────────────────────────────────
# NOTE: If you have access to qwen/qwq-32b, set JUDGE_MODEL to that.
# A bigger judge = more reliable scores. Generator model can stay small.
# ─────────────────────────────────────────────────────────────────────────────

CACHE_DIR = Path("../cache")
CACHE_DIR.mkdir(exist_ok=True)

def nim_call(prompt: str, model: str = GENERATOR_MODEL, max_tokens: int = 512) -> str:
    """
    Single NIM API call with retry logic.
    Sleeps 1s between calls automatically to avoid 429s on free tier.
    """
    for attempt in range(3):
        try:
            time.sleep(1)  # Rate limit buffer — DO NOT REMOVE
            response = nim.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = 5 * (attempt + 1)
            print(f"  ⚠ Attempt {attempt+1} failed: {e}. Retrying in {wait}s...")
            time.sleep(wait)
    return ""  # Return empty string on total failure — handled downstream

print("✅ Setup complete.")

---
## Part A — Build the Test Set

### What is a good test set?
Each entry has:
- `question` — a realistic question a policyholder would ask
- `ground_truth` — the correct answer (generated from the actual source chunk)
- `source_file` + `page` — where it came from (for debugging)

### Why generate ground truth from chunks?
You don't have time to manually write 20 answers. So we:
1. Sample a random chunk from your vector store
2. Ask NIM: *"Given this text, write one question AND its answer"*
3. The answer is grounded in the chunk text — so it's a valid reference

This is called **synthetic test set generation** and is standard practice.

### ⚠️ Run this ONCE — it saves to cache
The same test set is used for ALL your RAG strategies. This makes the comparison fair.

In [ ]:
# ── Load chunks from your Vector RAG cache ───────────────────────────────────
# (These are the same chunks from RAG_pipeline.ipynb)
chunks_file = CACHE_DIR / "chunks.pkl"

if not chunks_file.exists():
    raise FileNotFoundError(
        f"Chunks cache not found at {chunks_file}.\n"
        "Run RAG_pipeline.ipynb first to build and cache the chunks."
    )

with open(chunks_file, "rb") as f:
    chunks = pickle.load(f)

print(f"✅ Loaded {len(chunks)} chunks from cache")
print(f"   Sample chunk: {chunks[0].page_content[:120]}...")
print(f"   From file: {chunks[0].metadata.get('source_file', 'unknown')}")

In [ ]:
def generate_test_set(
    chunks,
    questions_per_pdf: int = 2,
    batch_size: int = 5,
    batch_pause: int = 10,
    seed: int = 42,
) -> list[dict]:
    """
    Generates question-answer pairs from chunks, 2 per PDF.

    Processes in batches of `batch_size` with a `batch_pause` second
    sleep between batches to avoid NVIDIA API rate limits.
    """
    random.seed(seed)

    # Group chunks by source PDF
    from collections import defaultdict
    pdf_chunks: dict[str, list] = defaultdict(list)
    for c in chunks:
        if len(c.page_content) > 200:
            src = c.metadata.get("source_file", "unknown")
            pdf_chunks[src].append(c)

    # Sample 2 chunks from each PDF
    selected = []
    for src, pool in pdf_chunks.items():
        pick = random.sample(pool, min(questions_per_pdf, len(pool)))
        selected.extend(pick)

    random.shuffle(selected)
    total = len(selected)
    print(f"Selected {total} chunks from {len(pdf_chunks)} PDFs ({questions_per_pdf} each)\n")

    test_set = []
    batch_num = 0

    for i, chunk in enumerate(selected, 1):
        # Pause between batches
        if i > 1 and (i - 1) % batch_size == 0:
            batch_num += 1
            print(f"\n⏸  Batch complete. Sleeping {batch_pause}s to avoid rate limits...\n")
            time.sleep(batch_pause)

        src_file = chunk.metadata.get("source_file", "?")
        page = chunk.metadata.get("page", "?")
        print(f"[{i}/{total}] Source: {src_file} p.{page}")

        prompt = f"""You are an insurance policy expert helping build a test set.

Read the policy text below carefully. Then:
1. Write ONE specific question that a real policyholder would ask about this text.
   - The question must be answerable from this text alone
   - Ask about a concrete detail: a number, a condition, a process, or a coverage rule
   - Do NOT ask vague questions like "what is insurance?"

2. Write the correct answer to that question.
   - Use only information from the text below
   - Be specific and complete (1-3 sentences)
   - Do not add information not present in the text

Policy text:
{chunk.page_content}

Respond in this EXACT JSON format (no extra text, no markdown):
{{"question": "...", "answer": "..."}}"""

        raw = nim_call(prompt, model=GENERATOR_MODEL, max_tokens=300)

        try:
            raw_clean = raw.replace("```json", "").replace("```", "").strip()
            parsed = json.loads(raw_clean)
            q = parsed.get("question", "").strip()
            a = parsed.get("answer", "").strip()

            if len(q) > 15 and len(a) > 20:
                test_set.append({
                    "question"    : q,
                    "ground_truth": a,
                    "source_chunk": chunk.page_content,
                    "source_file" : src_file,
                    "page"        : page,
                })
                print(f"  ✅ Q: {q[:80]}...")
            else:
                print(f"  ⚠ Bad parse, skipping")

        except json.JSONDecodeError:
            print(f"  ⚠ JSON decode failed. Raw: {raw[:80]}")

    print(f"\n✅ Generated {len(test_set)} test pairs")
    return test_set


# ── Run once, cache forever ───────────────────────────────────────────────────
testset_file = CACHE_DIR / "eval_testset.json"

if testset_file.exists():
    with open(testset_file) as f:
        test_set = json.load(f)
    print(f"✅ Loaded {len(test_set)} cached test pairs — skipping generation")
else:
    test_set = generate_test_set(chunks, questions_per_pdf=2, batch_size=5, batch_pause=10)
    with open(testset_file, "w") as f:
        json.dump(test_set, f, indent=2)
    print(f"\n✅ Saved to {testset_file}")

In [ ]:
# Preview the test set
print(f"Total questions: {len(test_set)}\n")
for i, item in enumerate(test_set[:5], 1):
    print(f"Q{i}: {item['question']}")
    print(f"   GT: {item['ground_truth'][:120]}...")
    print(f"   From: {item['source_file']} p.{item['page']}")
    print()

---
## Part B — Run Your Pipelines

For each RAG strategy, we call its query function on all 20 questions and collect:
- The generated **answer**
- The **retrieved contexts** (list of text chunks)

Each strategy saves its output to a JSON file. Run the right section for each pipeline.

### B1 — Vector RAG Pipeline Runner

In [ ]:
# ── Copy your Vector RAG setup from RAG_pipeline.ipynb ───────────────────────
# You need: embedding_manager, vector_store, rag_retriever, llm
# Paste those cell blocks here, or run RAG_pipeline.ipynb first
# and import from it if you prefer.
#
# Minimum needed:
#   from langchain_nvidia_ai_endpoints import ChatNVIDIA, NVIDIAEmbeddings
#   (EmbeddingManager, VectorStore, RAGRetriever, rag_query — paste here)

# PLACEHOLDER — replace with your actual imports
# from RAG_pipeline_setup import rag_retriever, llm, rag_query

print("⚠ Paste your Vector RAG setup code here before running this section.")

In [ ]:
def run_vector_rag_pipeline(test_set: list[dict]) -> list[dict]:
    """
    Runs Vector RAG on every question and collects outputs.
    """
    outputs = []
    print(f"Running Vector RAG on {len(test_set)} questions...\n")

    for i, item in enumerate(test_set, 1):
        q = item["question"]
        print(f"[{i}/{len(test_set)}] {q[:70]}...")

        try:
            # Direct retrieval for contexts
            retrieved = rag_retriever.retrieve(q, top_k=5, score_threshold=0.2)
            contexts = [doc["content"] for doc in retrieved]

            # Full pipeline for the answer
            result = rag_query(query=q, retriever=rag_retriever, llm=llm, top_k=5)
            answer = result.get("answer", "")
            category = result.get("category", "RELEVANT")

            outputs.append({
                "question"    : q,
                "ground_truth": item["ground_truth"],
                "contexts"    : contexts,
                "answer"      : answer,
                "category"    : category,
                "source_file" : item["source_file"],
            })
            print(f"  ✅ Answer: {answer[:80]}...")

        except Exception as e:
            print(f"  ❌ Error: {e}")
            outputs.append({
                "question"    : q,
                "ground_truth": item["ground_truth"],
                "contexts"    : [],
                "answer"      : f"ERROR: {e}",
                "category"    : "ERROR",
                "source_file" : item["source_file"],
            })

        time.sleep(1)  # Be gentle with NIM free tier

    return outputs


# ── Run and cache ─────────────────────────────────────────────────────────────
vector_rag_outputs_file = CACHE_DIR / "vector_rag_outputs.json"

if vector_rag_outputs_file.exists():
    with open(vector_rag_outputs_file) as f:
        vector_rag_outputs = json.load(f)
    print(f"✅ Loaded cached Vector RAG outputs ({len(vector_rag_outputs)} entries)")
else:
    vector_rag_outputs = run_vector_rag_pipeline(test_set)
    with open(vector_rag_outputs_file, "w") as f:
        json.dump(vector_rag_outputs, f, indent=2)
    print(f"✅ Saved Vector RAG outputs to {vector_rag_outputs_file}")

### B2 — PageIndex RAG Pipeline Runner

In [ ]:
# ── Copy your PageIndex setup from run_pageindex_v4.ipynb ────────────────────
# You need: pi_client, route_query, search_nodes, call_nim, REGISTRY_PATH

import pageindex.utils as _utils
from pageindex import PageIndexLocalClient

REGISTRY_PATH = "pdf_registry.json"

pi_client = PageIndexLocalClient(
    model="nvidia_nim/meta/llama-3.1-8b-instruct",
    summary_model="nvidia_nim/meta/llama-3.1-8b-instruct",
    retrieve_model="nvidia_nim/meta/llama-3.1-8b-instruct",
    storage_path=".pageindex",
)

# Paste route_query and search_nodes here (unchanged from run_pageindex_v4)
# ...

print("✅ PageIndex setup loaded.")

In [ ]:
def run_pageindex_pipeline(test_set: list[dict]) -> list[dict]:
    """
    Runs PageIndex RAG on every question and collects outputs.
    Forces routing to 'specific' for all questions so we always get retrieval.
    (Evaluation needs retrieved contexts — general/ambiguous routes give none.)
    """
    outputs = []
    print(f"Running PageIndex RAG on {len(test_set)} questions...\n")

    with open(REGISTRY_PATH) as f:
        registry = json.load(f)
    all_doc_ids = list(registry.keys())

    for i, item in enumerate(test_set, 1):
        q = item["question"]
        print(f"[{i}/{len(test_set)}] {q[:70]}...")

        try:
            routing = route_query(q)
            q_type = routing["type"]

            # For eval: if ambiguous or general, we still try to retrieve
            # from all documents (so we have contexts to score)
            if q_type in ("ambiguous", "general"):
                doc_ids_to_search = all_doc_ids[:3]  # search top 3 docs
            else:
                doc_ids_to_search = routing["doc_ids"]

            all_chunks = []
            for doc_id in doc_ids_to_search:
                chunks_found = search_nodes(doc_id, q)
                all_chunks.extend(chunks_found)

            if all_chunks:
                context = "\n\n---\n\n".join(all_chunks)
                answer = nim_call(
                    f"""Answer the question based only on the context below.
Question: {q}

Context:
{context}

Instructions:
- Use plain simple language
- Start with a one-sentence summary
- End with "Bottom line:" telling the user what to know""",
                    model=GENERATOR_MODEL,
                    max_tokens=512,
                )
            else:
                answer = "No relevant content found."

            outputs.append({
                "question"    : q,
                "ground_truth": item["ground_truth"],
                "contexts"    : all_chunks,
                "answer"      : answer,
                "q_type"      : q_type,
                "source_file" : item["source_file"],
            })
            print(f"  ✅ {len(all_chunks)} chunks | Answer: {answer[:60]}...")

        except Exception as e:
            print(f"  ❌ Error: {e}")
            outputs.append({
                "question"    : q,
                "ground_truth": item["ground_truth"],
                "contexts"    : [],
                "answer"      : f"ERROR: {e}",
                "q_type"      : "ERROR",
                "source_file" : item["source_file"],
            })

        time.sleep(1)

    return outputs


# ── Run and cache ─────────────────────────────────────────────────────────────
pageindex_outputs_file = CACHE_DIR / "pageindex_outputs.json"

if pageindex_outputs_file.exists():
    with open(pageindex_outputs_file) as f:
        pageindex_outputs = json.load(f)
    print(f"✅ Loaded cached PageIndex outputs ({len(pageindex_outputs)} entries)")
else:
    pageindex_outputs = run_pageindex_pipeline(test_set)
    with open(pageindex_outputs_file, "w") as f:
        json.dump(pageindex_outputs, f, indent=2)
    print(f"✅ Saved PageIndex outputs to {pageindex_outputs_file}")

### B3 — Graph RAG & Agentic RAG (Stubs)

When your teammate finishes Graph RAG, or when you build Agentic RAG, add their runner here.
All you need to do is: call their pipeline function, collect `contexts` and `answer`, save to JSON.
The evaluator in Part C works on any JSON with the same structure.

In [ ]:
# ── STUB: Graph RAG ───────────────────────────────────────────────────────────
# When ready, replace this with the actual runner.

def run_graph_rag_pipeline(test_set: list[dict]) -> list[dict]:
    outputs = []
    for item in test_set:
        # TODO: replace with actual Graph RAG call
        # graph_result = graph_rag_query(item["question"])
        outputs.append({
            "question"    : item["question"],
            "ground_truth": item["ground_truth"],
            "contexts"    : [],        # replace with graph_result["contexts"]
            "answer"      : "TODO",    # replace with graph_result["answer"]
            "source_file" : item["source_file"],
        })
    return outputs

print("Graph RAG stub defined — fill in when pipeline is ready.")

---
## Part C — NIM-as-Judge Evaluation

### How this works
For each question, we send 4 separate prompts to NIM (one per metric).
Each prompt asks NIM to score that metric from 0–1 and explain why.
We extract the score and save the reasoning for debugging.

### Rate limit math
20 questions × 4 metrics × 2 strategies = **160 NIM calls**
With `sleep(1)` that's ~3 minutes. No hangs, no 429s.

In [ ]:
# ── The 4 evaluation prompts ──────────────────────────────────────────────────
# Each returns JSON: {"score": float, "reasoning": str}

FAITHFULNESS_PROMPT = """You are an expert evaluator assessing whether an AI answer is faithful to its source context.

QUESTION: {question}

RETRIEVED CONTEXT:
{context}

GENERATED ANSWER:
{answer}

TASK:
1. List every factual claim made in the Generated Answer.
2. For each claim, determine if it is directly supported by the Retrieved Context.
3. Score = (number of supported claims) / (total claims). If there are no claims, score 1.0.

Respond in this EXACT JSON format only (no markdown, no extra text):
{{"score": 0.0, "reasoning": "claim 1: supported/not supported because... claim 2: ..."}}

Score must be between 0.0 and 1.0."""


ANSWER_RELEVANCY_PROMPT = """You are an expert evaluator assessing whether an AI answer is relevant to the question asked.

QUESTION: {question}

GENERATED ANSWER:
{answer}

TASK:
Score how directly and completely the answer addresses the question.
- 1.0 = answer directly addresses all parts of the question
- 0.7 = answer mostly relevant but misses a part or adds off-topic content
- 0.4 = answer is vaguely related but does not really answer the question
- 0.0 = answer is completely off-topic or refuses to answer

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "explanation of why this score was given"}}"""


CONTEXT_PRECISION_PROMPT = """You are an expert evaluator assessing the quality of retrieved context for a RAG system.

QUESTION: {question}

GROUND TRUTH ANSWER:
{ground_truth}

RETRIEVED CONTEXT CHUNKS:
{context_numbered}

TASK:
For each retrieved chunk, decide if it is relevant to answering the question (given what the ground truth says).
Score = (number of relevant chunks) / (total chunks).
If no chunks were retrieved, score is 0.0.

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "Chunk 1: relevant/not relevant because... Chunk 2: ..."}}"""


CONTEXT_RECALL_PROMPT = """You are an expert evaluator assessing whether a RAG system retrieved all necessary information.

QUESTION: {question}

GROUND TRUTH ANSWER:
{ground_truth}

RETRIEVED CONTEXT:
{context}

TASK:
1. List every key piece of information in the Ground Truth Answer.
2. For each key piece, check if it is present in the Retrieved Context.
3. Score = (pieces present in context) / (total key pieces).
If no context was retrieved, score is 0.0.

Respond in this EXACT JSON format only:
{{"score": 0.0, "reasoning": "Key point 1: found/not found in context... Key point 2: ..."}}"""


print("✅ Evaluation prompts defined.")

In [ ]:
def safe_parse_score(raw: str) -> dict:
    """
    Parses NIM's JSON response robustly.
    Handles cases where the model adds markdown fences or extra text.
    Returns {"score": float, "reasoning": str} or a fallback.
    """
    try:
        clean = raw.replace("```json", "").replace("```", "").strip()
        # Find the JSON object even if there's trailing text
        start = clean.find("{")
        end = clean.rfind("}") + 1
        if start == -1 or end == 0:
            raise ValueError("No JSON object found")
        parsed = json.loads(clean[start:end])
        score = float(parsed.get("score", 0.0))
        score = max(0.0, min(1.0, score))  # Clamp to [0, 1]
        return {"score": score, "reasoning": parsed.get("reasoning", "")}
    except Exception as e:
        return {"score": 0.0, "reasoning": f"Parse error: {e} | Raw: {raw[:100]}"}


def evaluate_single(item: dict) -> dict:
    """
    Evaluates one (question, contexts, answer, ground_truth) entry.
    Makes 4 NIM calls sequentially with sleep between each.
    Returns scores + reasoning for all 4 metrics.
    """
    q   = item["question"]
    a   = item["answer"]
    gt  = item["ground_truth"]
    ctx = item.get("contexts", [])

    # Build context strings for prompts
    context_joined = "\n\n".join(ctx) if ctx else "[No context retrieved]"
    context_numbered = "\n\n".join(
        f"[Chunk {i+1}]:\n{c}" for i, c in enumerate(ctx)
    ) if ctx else "[No context retrieved]"

    scores = {}

    # 1. Faithfulness
    raw = nim_call(
        FAITHFULNESS_PROMPT.format(question=q, context=context_joined, answer=a),
        model=JUDGE_MODEL, max_tokens=400
    )
    scores["faithfulness"] = safe_parse_score(raw)

    # 2. Answer Relevancy
    raw = nim_call(
        ANSWER_RELEVANCY_PROMPT.format(question=q, answer=a),
        model=JUDGE_MODEL, max_tokens=300
    )
    scores["answer_relevancy"] = safe_parse_score(raw)

    # 3. Context Precision
    raw = nim_call(
        CONTEXT_PRECISION_PROMPT.format(
            question=q, ground_truth=gt, context_numbered=context_numbered
        ),
        model=JUDGE_MODEL, max_tokens=400
    )
    scores["context_precision"] = safe_parse_score(raw)

    # 4. Context Recall
    raw = nim_call(
        CONTEXT_RECALL_PROMPT.format(question=q, ground_truth=gt, context=context_joined),
        model=JUDGE_MODEL, max_tokens=400
    )
    scores["context_recall"] = safe_parse_score(raw)

    return scores


def evaluate_pipeline(pipeline_outputs: list[dict], strategy_name: str) -> list[dict]:
    """
    Evaluates all outputs of one RAG strategy.
    Returns a list of per-question results with scores.
    """
    results = []
    total = len(pipeline_outputs)
    print(f"\n{'='*60}")
    print(f"Evaluating: {strategy_name} ({total} questions × 4 metrics = {total*4} NIM calls)")
    print(f"{'='*60}")

    for i, item in enumerate(pipeline_outputs, 1):
        print(f"\n[{i}/{total}] {item['question'][:65]}...")

        # Skip entries that errored during pipeline run
        if item["answer"].startswith("ERROR") or item["answer"] == "TODO":
            print("  ⏭ Skipped (pipeline error)")
            continue

        scores = evaluate_single(item)

        result = {
            "question"        : item["question"],
            "answer"          : item["answer"],
            "ground_truth"    : item["ground_truth"],
            "n_contexts"      : len(item.get("contexts", [])),
            "faithfulness"    : scores["faithfulness"]["score"],
            "answer_relevancy": scores["answer_relevancy"]["score"],
            "context_precision": scores["context_precision"]["score"],
            "context_recall"  : scores["context_recall"]["score"],
            "reasoning"       : scores,
        }
        results.append(result)

        # Print live scores
        print(f"  F={result['faithfulness']:.2f}  "
              f"AR={result['answer_relevancy']:.2f}  "
              f"CP={result['context_precision']:.2f}  "
              f"CR={result['context_recall']:.2f}")

    print(f"\n✅ Done: {len(results)}/{total} questions evaluated")
    return results


print("✅ Evaluator functions defined.")

In [ ]:
# ── Evaluate Vector RAG ───────────────────────────────────────────────────────
vector_eval_file = CACHE_DIR / "vector_rag_eval_results.json"

if vector_eval_file.exists():
    with open(vector_eval_file) as f:
        vector_eval_results = json.load(f)
    print(f"✅ Loaded cached Vector RAG eval results ({len(vector_eval_results)} entries)")
else:
    vector_eval_results = evaluate_pipeline(vector_rag_outputs, "Vector RAG (ChromaDB)")
    with open(vector_eval_file, "w") as f:
        json.dump(vector_eval_results, f, indent=2)
    print(f"✅ Saved to {vector_eval_file}")

In [ ]:
# ── Evaluate PageIndex RAG ────────────────────────────────────────────────────
pageindex_eval_file = CACHE_DIR / "pageindex_eval_results.json"

if pageindex_eval_file.exists():
    with open(pageindex_eval_file) as f:
        pageindex_eval_results = json.load(f)
    print(f"✅ Loaded cached PageIndex eval results ({len(pageindex_eval_results)} entries)")
else:
    pageindex_eval_results = evaluate_pipeline(pageindex_outputs, "PageIndex RAG")
    with open(pageindex_eval_file, "w") as f:
        json.dump(pageindex_eval_results, f, indent=2)
    print(f"✅ Saved to {pageindex_eval_file}")

---
## Part D — Results & Comparison Table

In [ ]:
import pandas as pd

METRICS = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]

def compute_averages(results: list[dict], strategy_name: str) -> dict:
    if not results:
        return {"Strategy": strategy_name, **{m: None for m in METRICS}}
    df = pd.DataFrame(results)
    avgs = df[METRICS].mean().round(3)
    return {"Strategy": strategy_name, **avgs.to_dict()}


# Build comparison table
summary_rows = [
    compute_averages(vector_eval_results,   "Vector RAG (ChromaDB)"),
    compute_averages(pageindex_eval_results, "PageIndex RAG"),
    # Add Graph RAG and Agentic RAG rows here when ready:
    # compute_averages(graph_eval_results,   "Graph RAG"),
    # compute_averages(agentic_eval_results, "Agentic RAG"),
]

summary_df = pd.DataFrame(summary_rows).set_index("Strategy")

print("\n" + "="*65)
print("  COMPARATIVE RAG EVALUATION RESULTS")
print("="*65)
print(summary_df.to_string())
print("="*65)
print("  All scores 0.0 – 1.0   |   Higher is better")
print("  F=Faithfulness | AR=Answer Relevancy | CP=Context Precision | CR=Context Recall")

In [ ]:
# ── Export for LaTeX report ───────────────────────────────────────────────────
summary_df.to_csv(CACHE_DIR / "rag_comparison_table.csv")
print(f"✅ Table saved to {CACHE_DIR / 'rag_comparison_table.csv'}")

# LaTeX table snippet
print("\n── LaTeX snippet for your report ──")
print(summary_df.to_latex(float_format="{:.3f}".format))

In [ ]:
# ── Per-question breakdown for worst-performers ───────────────────────────────
def show_worst(results: list[dict], strategy: str, metric: str, n: int = 3):
    df = pd.DataFrame(results)
    worst = df.nsmallest(n, metric)[["question", metric, "reasoning"]]
    print(f"\n❌ {strategy} — lowest {metric} questions:")
    for _, row in worst.iterrows():
        print(f"  Q: {row['question'][:70]}")
        print(f"  Score: {row[metric]:.3f}")
        reasoning = row['reasoning'].get(metric, {}).get('reasoning', '')
        print(f"  Why: {reasoning[:150]}...")
        print()

show_worst(vector_eval_results,   "Vector RAG",   "faithfulness")
show_worst(vector_eval_results,   "Vector RAG",   "context_recall")
show_worst(pageindex_eval_results, "PageIndex RAG", "faithfulness")
show_worst(pageindex_eval_results, "PageIndex RAG", "context_recall")

---
## Troubleshooting Guide

### All scores are 0.0
→ `safe_parse_score` is returning fallbacks. Print the raw NIM output to see what's happening:
```python
raw = nim_call(FAITHFULNESS_PROMPT.format(...), model=JUDGE_MODEL)
print(repr(raw))  # See the actual response
```

### NIM returns 429 errors
→ Increase `time.sleep()` in `nim_call` from 1 to 3 seconds.
→ Or spread runs across two sessions using the JSON cache.

### Context Precision is very low
→ Your retriever is pulling noisy chunks. For Vector RAG: increase `score_threshold` from 0.2 to 0.4.
→ For PageIndex: the node selector LLM is picking too many irrelevant nodes.

### Context Recall is very low
→ Your retriever is missing relevant chunks. For Vector RAG: increase `top_k` from 5 to 8.
→ For PageIndex: the routing LLM may be selecting the wrong document.

### Faithfulness is low
→ The generator LLM is ignoring context and using its own knowledge.
→ Strengthen the system prompt: "Answer ONLY from the context. If not in context, say you don't know."

### Adding Graph RAG / Agentic RAG later
1. Build a `run_X_pipeline()` function following the B1/B2 pattern
2. Save output to `CACHE_DIR / "X_outputs.json"`
3. Call `evaluate_pipeline(X_outputs, "X RAG")`
4. Add a row to `summary_rows` in Part D